In [1]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}


In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup


In [3]:
import requests
from bs4 import BeautifulSoup

url = "https://www.globalfirepower.com/total-population-by-country.php"
r = requests.get(url, headers=headers)
soup = BeautifulSoup(r.text, "html.parser")

print(len(r.text))


319442


In [4]:
table = soup.find("table")
print(table)


None


In [5]:
blocks = soup.find_all("div", class_="col-md-4 col-sm-6")
len(blocks)


0

In [6]:
rows = soup.find_all("div", class_="topRow")
len(rows)


145

In [7]:
rows[0]


<div class="topRow">
<div class="rankNumContainer">
<span class="textWhite textLarge textBold">
					1                </span>
</div>
<div class="countryNameContainer">
<div class="longFormName">
<span class="textWhite textLarge textShadow">
						China                    </span>
</div>
<div class="shortFormName">
<span class="textWhite textLarge textShadow">
						CHN                    </span>
</div>
</div>
<div class="valueContainer">
<span class="textWhite textLarge">
<span style="background-color:#000; padding:3px 7px 3px 7px; border-bottom:thin solid #666;">
													1,415,043,270                                            </span>
</span>
</div>
</div>

In [8]:
row = rows[0]

for child in row.find_all("div"):
    print(child.text.strip())


1
China                    



						CHN
China
CHN
1,415,043,270


In [9]:
data = []

for row in rows:
    cols = row.find_all("div")
    
    country = cols[1].text.strip()
    value = cols[-1].text.strip()

    data.append({
        "country": country,
        "population": value
    })

data[:5]


[{'country': 'China                    \n\n\n\r\n\t\t\t\t\t\tCHN',
  'population': '1,415,043,270'},
 {'country': 'India                    \n\n\n\r\n\t\t\t\t\t\tIND',
  'population': '1,409,128,296'},
 {'country': 'United States                    \n\n\n\r\n\t\t\t\t\t\tUSA',
  'population': '341,963,408'},
 {'country': 'Indonesia                    \n\n\n\r\n\t\t\t\t\t\tINO',
  'population': '281,562,465'},
 {'country': 'Pakistan                    \n\n\n\r\n\t\t\t\t\t\tPAK',
  'population': '252,363,571'}]

In [10]:
cleaned_data = []

for item in data:
    clean_country = item["country"].split("\n")[0].strip()

    cleaned_data.append({
        "country": clean_country,
        "population": item["population"]
    })

cleaned_data[:5]


[{'country': 'China', 'population': '1,415,043,270'},
 {'country': 'India', 'population': '1,409,128,296'},
 {'country': 'United States', 'population': '341,963,408'},
 {'country': 'Indonesia', 'population': '281,562,465'},
 {'country': 'Pakistan', 'population': '252,363,571'}]

In [11]:
import pandas as pd

df_population = pd.DataFrame(cleaned_data)
df_population.head()


,country,population
0,China,"1,415,043,270"
1,India,"1,409,128,296"
2,United States,"341,963,408"
3,Indonesia,"281,562,465"
4,Pakistan,"252,363,571"


In [12]:
df_population.to_csv("../data/population.csv", index=False)
print("population.csv saved")


population.csv saved


In [13]:
def scrape_single_metric(url, metric_name):
    import requests
    from bs4 import BeautifulSoup
    import pandas as pd

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    rows = soup.find_all("div", class_="topRow")

    data = []

    for row in rows:
        cols = row.find_all("div")
        country = cols[1].text.split("\n")[0].strip()
        value = cols[-1].text.strip()

        data.append({
            "country": country,
            metric_name: value
        })

    return pd.DataFrame(data)


In [14]:
manpower_url = "https://www.globalfirepower.com/available-military-manpower.php"

df_manpower = scrape_single_metric(
    manpower_url,
    "total_military_manpower"
)

df_manpower.head()


,country,total_military_manpower
0,,"764,123,366"
1,,"662,290,299"
2,,"150,463,900"
3,,"137,965,608"
4,,"125,475,979"


In [60]:
# DEBUG: active personnel page structure
rows = BeautifulSoup(
    requests.get(active_url, headers={"User-Agent": "Mozilla/5.0"}).text,
    "html.parser"
).find_all("div", class_="topRow")

row = rows[0]
for i, div in enumerate(row.find_all("div")):
    print(i, "=>", div.text.strip())


0 => 1
1 => United States                    



						USA
2 => United States
3 => USA
4 => 20,879,000                        						
							bbl


In [ ]:
def scrape_single_metric(url, metric_name):
    import requests
    from bs4 import BeautifulSoup
    import pandas as pd

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    rows = soup.find_all("div", class_="topRow")
    data = []

    for row in rows:
        cols = row.find_all("div")

        if len(cols) >= 5:
            country = cols[2].text.strip()
            value = cols[4].text.strip()

            data.append({
                "country": country,
                metric_name: value
            })

    return pd.DataFrame(data)


In [ ]:
df_active.head()

In [ ]:
# Re-load manpower page fresh
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

manpower_url = "https://www.globalfirepower.com/available-military-manpower.php"
soup = BeautifulSoup(
    requests.get(manpower_url, headers=headers).text,
    "html.parser"
)

rows = soup.find_all("div", class_="topRow")

# Inspect first row again
row = rows[0]
for i, div in enumerate(row.find_all("div")):
    print(i, "=>", div.text.strip())


In [16]:
# Correct manpower extraction (NO function, direct code)

data = []

for row in rows:
    cols = row.find_all("div")

    if len(cols) >= 5:
        country = cols[2].text.strip()
        value = cols[4].text.strip()

        data.append({
            "country": country,
            "total_military_manpower": value
        })

data[:5]


[{'country': 'China', 'total_military_manpower': '1,415,043,270'},
 {'country': 'India', 'total_military_manpower': '1,409,128,296'},
 {'country': 'United States', 'total_military_manpower': '341,963,408'},
 {'country': 'Indonesia', 'total_military_manpower': '281,562,465'},
 {'country': 'Pakistan', 'total_military_manpower': '252,363,571'}]

In [17]:
import pandas as pd

df_manpower = pd.DataFrame(data)
df_manpower.head()


,country,total_military_manpower
0,China,"1,415,043,270"
1,India,"1,409,128,296"
2,United States,"341,963,408"
3,Indonesia,"281,562,465"
4,Pakistan,"252,363,571"


In [18]:
df_manpower.to_csv("../data/total_military_manpower.csv", index=False)
print("total_military_manpower.csv saved")


total_military_manpower.csv saved


In [19]:
# Load active personnel page fresh
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

active_url = "https://www.globalfirepower.com/active-military-manpower.php"
soup = BeautifulSoup(
    requests.get(active_url, headers=headers).text,
    "html.parser"
)

rows = soup.find_all("div", class_="topRow")
len(rows)


145

In [20]:
row = rows[0]

for i, div in enumerate(row.find_all("div")):
    print(i, "=>", div.text.strip())


0 => 1
1 => China                    



						CHN
2 => China
3 => CHN
4 => 2,035,000


In [21]:
data_active = []

for row in rows:
    cols = row.find_all("div")

    if len(cols) >= 5:
        country = cols[2].text.strip()   # CLEAN country
        value = cols[4].text.strip()     # ACTIVE personnel

        data_active.append({
            "country": country,
            "active_personnel": value
        })

data_active[:5]


[{'country': 'China', 'active_personnel': '2,035,000'},
 {'country': 'India', 'active_personnel': '1,455,550'},
 {'country': 'United States', 'active_personnel': '1,328,000'},
 {'country': 'North Korea', 'active_personnel': '1,320,000'},
 {'country': 'Russia', 'active_personnel': '1,320,000'}]

In [22]:
df_active = pd.DataFrame(data_active)
df_active.head()


,country,active_personnel
0,China,"2,035,000"
1,India,"1,455,550"
2,United States,"1,328,000"
3,North Korea,"1,320,000"
4,Russia,"1,320,000"


In [23]:
df_active.to_csv("../data/active_personnel.csv", index=False)
print("active_personnel.csv saved")


active_personnel.csv saved


In [24]:
row = rows[0]

for i, div in enumerate(row.find_all("div")):
    print(i, "=>", div.text.strip())


0 => 1
1 => China                    



						CHN
2 => China
3 => CHN
4 => 2,035,000


In [25]:
# Load active personnel page fresh
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

active_url = "https://www.globalfirepower.com/aircraft-total-fighters.php"
soup = BeautifulSoup(
    requests.get(active_url, headers=headers).text,
    "html.parser"
)

rows = soup.find_all("div", class_="topRow")
len(rows)


145

In [26]:
row = rows[0]

for i, div in enumerate(row.find_all("div")):
    print(i, "=>", div.text.strip())


0 => 1
1 => United States                    



						USA
2 => United States
3 => USA
4 => 1,790


In [27]:
data_active = []

for row in rows:
    cols = row.find_all("div")

    if len(cols) >= 5:
        country = cols[2].text.strip()   # CLEAN country
        value = cols[4].text.strip()     # ACTIVE personnel

        data_active.append({
            "country": country,
            "fighter_aircraft": value
        })

data_active[:5]


[{'country': 'United States', 'fighter_aircraft': '1,790'},
 {'country': 'China', 'fighter_aircraft': '1,212'},
 {'country': 'Russia', 'fighter_aircraft': '833'},
 {'country': 'India', 'fighter_aircraft': '513'},
 {'country': 'North Korea', 'fighter_aircraft': '368'}]

In [28]:
df_active = pd.DataFrame(data_active)
df_active.head()


,country,fighter_aircraft
0,United States,"1,790"
1,China,"1,212"
2,Russia,833
3,India,513
4,North Korea,368


In [29]:
df_active.to_csv("../data/fighter_aircraft.csv", index=False)
print("fighter_aircraft.csv saved")


fighter_aircraft.csv saved


In [30]:
# Load active personnel page fresh
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

active_url = "https://www.globalfirepower.com/armor-tanks-total.php"
soup = BeautifulSoup(
    requests.get(active_url, headers=headers).text,
    "html.parser"
)

rows = soup.find_all("div", class_="topRow")
len(rows)


145

In [31]:
row = rows[0]

for i, div in enumerate(row.find_all("div")):
    print(i, "=>", div.text.strip())


0 => 1
1 => China                    



						CHN
2 => China
3 => CHN
4 => 6,800


In [32]:
data_active = []

for row in rows:
    cols = row.find_all("div")

    if len(cols) >= 5:
        country = cols[2].text.strip()   # CLEAN country
        value = cols[4].text.strip()     # ACTIVE personnel

        data_active.append({
            "country": country,
            "tanks": value
        })

data_active[:5]


[{'country': 'China', 'tanks': '6,800'},
 {'country': 'Russia', 'tanks': '5,750'},
 {'country': 'United States', 'tanks': '4,640'},
 {'country': 'North Korea', 'tanks': '4,344'},
 {'country': 'India', 'tanks': '4,201'}]

In [33]:
df_active = pd.DataFrame(data_active)
df_active.head()


,country,tanks
0,China,"6,800"
1,Russia,"5,750"
2,United States,"4,640"
3,North Korea,"4,344"
4,India,"4,201"


In [34]:
df_active.to_csv("../data/tanks.csv", index=False)
print("tanks.csv saved")


tanks.csv saved


In [35]:
# Load active personnel page fresh
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

active_url = "https://www.globalfirepower.com/navy-submarines.php"
soup = BeautifulSoup(
    requests.get(active_url, headers=headers).text,
    "html.parser"
)

rows = soup.find_all("div", class_="topRow")
len(rows)


145

In [36]:
row = rows[0]

for i, div in enumerate(row.find_all("div")):
    print(i, "=>", div.text.strip())


0 => 1
1 => United States                    



						USA
2 => United States
3 => USA
4 => 70


In [37]:
data_active = []

for row in rows:
    cols = row.find_all("div")

    if len(cols) >= 5:
        country = cols[2].text.strip()   # CLEAN country
        value = cols[4].text.strip()     # ACTIVE personnel

        data_active.append({
            "country": country,
            "submarines": value
        })

data_active[:5]


[{'country': 'United States', 'submarines': '70'},
 {'country': 'Russia', 'submarines': '63'},
 {'country': 'China', 'submarines': '61'},
 {'country': 'Iran', 'submarines': '25'},
 {'country': 'Japan', 'submarines': '24'}]

In [38]:
df_active = pd.DataFrame(data_active)
df_active.head()


,country,submarines
0,United States,70
1,Russia,63
2,China,61
3,Iran,25
4,Japan,24


In [39]:
df_active.to_csv("../data/submarine.csv", index=False)
print("submarine.csv saved")


submarine.csv saved


In [64]:
import re

usd_to_inr = 83.0  # approx rate

data_active = []

for row in rows:
    cols = row.find_all("div")

    if len(cols) >= 5:
        country = cols[2].text.strip()
        raw_text = cols[4].text.strip()

        # Extract numeric part only (digits + commas)
        match = re.search(r'[\d,]+', raw_text)

        if match:
            value_usd = float(match.group().replace(",", ""))
            value_inr_crore = (value_usd * usd_to_inr) / 1e7
            value_inr_crore = round(value_inr_crore, 2)
        else:
            value_inr_crore = None   # 👈 very important

        data_active.append({
            "country": country,
            "defense_budget_inr_crore": value_inr_crore
        })

data_active[:5]


[{'country': 'United States', 'defense_budget_inr_crore': 173.3},
 {'country': 'Saudi Arabia', 'defense_budget_inr_crore': 92.24},
 {'country': 'Russia', 'defense_budget_inr_crore': 89.03},
 {'country': 'Canada', 'defense_budget_inr_crore': 47.24},
 {'country': 'China', 'defense_budget_inr_crore': 41.37}]

In [65]:
df_active = pd.DataFrame(data_active)
df_active.head()


,country,defense_budget_inr_crore
0,United States,173.30
1,Saudi Arabia,92.24
2,Russia,89.03
3,Canada,47.24
4,China,41.37


In [66]:
df_active.to_csv("../data/defense_budget_usd_billion.csv", index=False)
print("defense_budget_usd_billion.csv saved")


defense_budget_usd_billion.csv saved


In [43]:
# Load active personnel page fresh
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

active_url = "https://www.globalfirepower.com/major-serviceable-airports-by-country.php"
soup = BeautifulSoup(
    requests.get(active_url, headers=headers).text,
    "html.parser"
)

rows = soup.find_all("div", class_="topRow")
len(rows)


145

In [44]:
row = rows[0]

for i, div in enumerate(row.find_all("div")):
    print(i, "=>", div.text.strip())


0 => 1
1 => United States                    



						USA
2 => United States
3 => USA
4 => 15,873


In [45]:
data_active = []

for row in rows:
    cols = row.find_all("div")

    if len(cols) >= 5:
        country = cols[2].text.strip()   # CLEAN country
        value = cols[4].text.strip()     # ACTIVE personnel

        data_active.append({
            "country": country,
            "total_serviceable_airports": value
        })

data_active[:5]


[{'country': 'United States', 'total_serviceable_airports': '15,873'},
 {'country': 'Brazil', 'total_serviceable_airports': '4,919'},
 {'country': 'Australia', 'total_serviceable_airports': '2,180'},
 {'country': 'Mexico', 'total_serviceable_airports': '1,485'},
 {'country': 'Canada', 'total_serviceable_airports': '1,425'}]

In [46]:
df_active = pd.DataFrame(data_active)
df_active.head()


,country,total_serviceable_airports
0,United States,"15,873"
1,Brazil,"4,919"
2,Australia,"2,180"
3,Mexico,"1,485"
4,Canada,"1,425"


In [47]:
df_active.to_csv("../data/total_serviceable_airports.csv", index=False)
print("total_serviceable_airports.csv saved")


total_serviceable_airports.csv saved


In [52]:
# Load active personnel page fresh
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

active_url = "https://www.globalfirepower.com/oil-production-by-country.php"
soup = BeautifulSoup(
    requests.get(active_url, headers=headers).text,
    "html.parser"
)

rows = soup.find_all("div", class_="topRow")
len(rows)


145

In [53]:
row = rows[0]

for i, div in enumerate(row.find_all("div")):
    print(i, "=>", div.text.strip())


0 => 1
1 => United States                    



						USA
2 => United States
3 => USA
4 => 20,879,000                        						
							bbl


In [55]:
data_active = []

for row in rows:
    cols = row.find_all("div")

    if len(cols) >= 5:
        country = cols[2].text.strip()

        value_text = (
            cols[4].text
            .strip()
            .replace("\t", "")
            .replace("\n", "")
            .replace(",", "")
        )

        try:
            value_bbl = float(value_text)
            value_million_bbl = value_bbl / 1e6
            value_million_bbl = round(value_million_bbl, 2)
        except:
            value_million_bbl = None

        data_active.append({
            "country": country,
            "oil_production_million_bbl": value_million_bbl
        })

data_active[:5]


[{'country': 'United States', 'oil_production_million_bbl': None},
 {'country': 'Saudi Arabia', 'oil_production_million_bbl': None},
 {'country': 'Russia', 'oil_production_million_bbl': None},
 {'country': 'Canada', 'oil_production_million_bbl': None},
 {'country': 'China', 'oil_production_million_bbl': None}]

[{'country': 'United States',
  'oil_production_million_bbl': '20,879,000                        \t\t\t\t\t\t\r\n\t\t\t\t\t\t\tbbl'},
 {'country': 'Saudi Arabia',
  'oil_production_million_bbl': '11,113,000                        \t\t\t\t\t\t\r\n\t\t\t\t\t\t\tbbl'},
 {'country': 'Russia',
  'oil_production_million_bbl': '10,727,000                        \t\t\t\t\t\t\r\n\t\t\t\t\t\t\tbbl'},
 {'country': 'Canada',
  'oil_production_million_bbl': '5,692,000                        \t\t\t\t\t\t\r\n\t\t\t\t\t\t\tbbl'},
 {'country': 'China',
  'oil_production_million_bbl': '4,984,000                        \t\t\t\t\t\t\r\n\t\t\t\t\t\t\tbbl'}]

In [57]:
df_active = pd.DataFrame(data_active)
df_active.head()


,country,oil_production_million_bbl
0,United States,"20,879,000 \t\t\t\t\t\t..."
1,Saudi Arabia,"11,113,000 \t\t\t\t\t\t..."
2,Russia,"10,727,000 \t\t\t\t\t\t..."
3,Canada,"5,692,000 \t\t\t\t\t\t\..."
4,China,"4,984,000 \t\t\t\t\t\t\..."


In [70]:
import pandas as pd
from functools import reduce

df_population = pd.read_csv("../data/population.csv")
df_manpower = pd.read_csv("../data/total_military_manpower.csv")
df_active = pd.read_csv("../data/active_personnel.csv")
df_budget = pd.read_csv("../data/defense_budget_usd_billion.csv")
df_fighter = pd.read_csv("../data/fighter_aircraft.csv")
df_tanks = pd.read_csv("../data/tanks.csv")
df_submarine = pd.read_csv("../data/submarine.csv")
df_airports = pd.read_csv("../data/total_serviceable_airports.csv")

df_population.head()


,country,population
0,China,"1,415,043,270"
1,India,"1,409,128,296"
2,United States,"341,963,408"
3,Indonesia,"281,562,465"
4,Pakistan,"252,363,571"


In [71]:
dfs = [
    df_population,
    df_manpower,
    df_active,
    df_budget,
    df_fighter,
    df_tanks,
    df_submarine,
    df_airports
]

df_master = reduce(
    lambda left, right: pd.merge(left, right, on="country", how="left"),
    dfs
)

df_master.head()


,country,population,total_military_manpower,active_personnel,defense_budget_inr_crore,fighter_aircraft,tanks,submarines,total_serviceable_airports
0,China,"1,415,043,270","1,415,043,270","2,035,000",41.37,"1,212","6,800",61,531
1,India,"1,409,128,296","1,409,128,296","1,455,550",6.60,513,"4,201",18,311
2,United States,"341,963,408","341,963,408","1,328,000",173.30,"1,790","4,640",70,"15,873"
3,Indonesia,"281,562,465","281,562,465","400,000",7.18,41,331,4,513
4,Pakistan,"252,363,571","252,363,571","654,000",0.84,328,"2,627",8,116


In [72]:
for col in df_master.columns:
    if col != "country":
        df_master[col] = (
            df_master[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("Billion", "", regex=False)
            .str.strip()
        )

df_master.iloc[:, 1:] = df_master.iloc[:, 1:].apply(
    pd.to_numeric, errors="coerce"
)

df_master.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   country                     145 non-null    object
 1   population                  145 non-null    object
 2   total_military_manpower     145 non-null    object
 3   active_personnel            145 non-null    object
 4   defense_budget_inr_crore    145 non-null    object
 5   fighter_aircraft            145 non-null    object
 6   tanks                       145 non-null    object
 7   submarines                  145 non-null    object
 8   total_serviceable_airports  145 non-null    object
dtypes: object(9)
memory usage: 10.3+ KB


In [73]:
df_master.to_csv("../data/master_military_data.csv", index=False)
print("✅ master_military_data.csv saved")


✅ master_military_data.csv saved
